# Inter-Annotator Agreement (IAA)

Compare manual labels from two annotators on the same `Review_ID` rows: **Subjectivity** and **Polarity**.

Inputs:
- `Hand_Labeling_Task_Trurong.xlsx`
- `Hand_Labeling_Task_Huiling.xlsx`

In [ ]:
import pandas as pd
from sklearn.metrics import cohen_kappa_score

print("pandas:", pd.__version__)

## 1. Data preparation

Load both files, align rows by `Review_ID`, and extract `Subjectivity` and `Polarity` from each annotator.

In [3]:
FILE_A = "Hand_Labeling_Task_Truong.xlsx"
FILE_B = "Hand_Labeling_Task_Huiling.xlsx"

df_a = pd.read_excel(FILE_A, engine="openpyxl")
df_b = pd.read_excel(FILE_B, engine="openpyxl")

required = {"Review_ID", "Subjectivity", "Polarity"}
for name, d in [(FILE_A, df_a), (FILE_B, df_b)]:
    missing = required - set(d.columns)
    if missing:
        raise ValueError(f"{name} is missing columns: {sorted(missing)}")

# Align by Review_ID (inner join keeps only IDs present in both)
merged = df_a.merge(
    df_b,
    on="Review_ID",
    how="inner",
    suffixes=("_Trurong", "_Huiling"),
    validate="one_to_one",
)

n = len(merged)
print(f"Aligned rows (inner join on Review_ID): {n}")

if n != 1000:
    print(
        f"Warning: expected 1,000 aligned rows but got {n}. "
        "Check for duplicate Review_ID or mismatched files."
    )

subj_tr = merged["Subjectivity_Trurong"]
subj_hu = merged["Subjectivity_Huiling"]
pol_tr = merged["Polarity_Trurong"]
pol_hu = merged["Polarity_Huiling"]

Aligned rows (inner join on Review_ID): 1000


## 2. Metrics

- **Simple percentage agreement**: fraction of rows where both labels match.
- **Cohen's kappa**: `sklearn.metrics.cohen_kappa_score` for each task.

In [4]:
def simple_agreement(a, b, denom: int):
    a = pd.to_numeric(pd.Series(a), errors="coerce")
    b = pd.to_numeric(pd.Series(b), errors="coerce")
    same = (a == b) & a.notna() & b.notna()
    return same.sum() / denom if denom else float("nan")


def kappa_safe(y1, y2):
    y1 = pd.Series(y1)
    y2 = pd.Series(y2)
    mask = y1.notna() & y2.notna()
    if mask.sum() == 0:
        return float("nan")
    return cohen_kappa_score(y1[mask], y2[mask])


denom = n  # identical labels / total aligned rows (use 1000 when both files align to 1000 rows)
pct_subj = simple_agreement(subj_tr, subj_hu, denom) * 100
pct_pol = simple_agreement(pol_tr, pol_hu, denom) * 100
kappa_subj = kappa_safe(subj_tr, subj_hu)
kappa_pol = kappa_safe(pol_tr, pol_hu)

## 3. Interpretation summary

In [5]:
print(f"Simple Agreement (Subjectivity): {pct_subj:.2f}%")
print(f"Cohen's Kappa (Subjectivity): {kappa_subj:.4f}")
print(f"Simple Agreement (Polarity): {pct_pol:.2f}%")
print(f"Cohen's Kappa (Polarity): {kappa_pol:.4f}")

Simple Agreement (Subjectivity): 93.30%
Cohen's Kappa (Subjectivity): 0.2913
Simple Agreement (Polarity): 91.10%
Cohen's Kappa (Polarity): 0.8027


## 4. Disagreements export

Rows where **Subjectivity** or **Polarity** (or both) differ between annotators.

In [6]:
OUT_XLSX = "Labeling_Discrepancies.xlsx"

subj_disagree = subj_tr != subj_hu
pol_disagree = pol_tr != pol_hu
any_disagree = subj_disagree | pol_disagree

text_src = "Review_Text_Trurong" if "Review_Text_Trurong" in merged.columns else None

discrepancies = merged.loc[any_disagree].copy()
if text_src:
    discrepancies["Original_Text"] = discrepancies[text_src]
out_cols = [
    "Review_ID",
    "Original_Text",
    "Subjectivity_Trurong",
    "Subjectivity_Huiling",
    "Polarity_Trurong",
    "Polarity_Huiling",
]
out_cols = [c for c in out_cols if c in discrepancies.columns]
discrepancies = discrepancies[out_cols]
discrepancies["Human_Correctness_Check"] = ""

discrepancies.to_excel(OUT_XLSX, index=False, engine="openpyxl")
print(f"Disagreement rows: {len(discrepancies)} (saved to {OUT_XLSX})")
discrepancies.head(10)

Disagreement rows: 112 (saved to Labeling_Discrepancies.xlsx)


,Review_ID,Original_Text,Subjectivity_Trurong,Subjectivity_Huiling,Polarity_Trurong,Polarity_Huiling,Human_Correctness_Check
15,RV03334,Service was bad. Location is good. Soup broth ...,1,1,-1,1,
21,RV06479,The milo here tastes like plain water. Never o...,1,1,-1,1,
24,RV07202,I am disappointed with the mee siam I purchase...,1,0,-1,0,
34,RV01852,One of the pioneer hawkers at tourist-packed M...,1,1,1,-1,
35,RV09783,Many places cant get laksa right because their...,1,1,-1,0,
43,RV00741,A lot of fish but soup is the light soup kind\...,1,0,1,0,
44,RV04327,"😋 Whenever I'm in the area, I make a point to ...",1,1,-1,0,
45,RV05512,Came here on a weekend and their business was ...,1,1,1,-1,
51,RV03061,The food price is quite reasonable for the amo...,1,1,1,-1,
53,RV03412,Ordered a kopi si siu dai (SGD1.70 in a takeaw...,1,1,-1,0,
